# 05 — Query Transformations

*Level 2 — Advanced RAG*

## Objective
Rewrite, expand, and reframe a query with a local LLM (Ollama) before retrieval, and see the effect on what gets retrieved.


In [1]:
import sys
from pathlib import Path

LEVEL_DIR = Path.cwd().parent
sys.path.insert(0, str(LEVEL_DIR))
sys.path.insert(0, str(LEVEL_DIR / "hybrid-search"))
sys.path.insert(0, str(LEVEL_DIR / "query-transformations"))
sys.path.insert(0, str(LEVEL_DIR / "metadata-filtering"))
sys.path.insert(0, str(LEVEL_DIR / "context-compression"))


In [2]:
from common.dataset import prepare
from retrieval.dense import DenseRetriever

data = prepare()
corpus_texts = {d: data.corpus_text(d) for d in data.doc_ids()}
dense = DenseRetriever.from_corpus(corpus_texts)

sample_qid = next(iter(data.qrels))
query = data.queries[sample_qid]
relevant = set(data.qrels[sample_qid])
print(f"Query: {query!r}\nRelevant doc(s): {relevant}")


Query: '0-dimensional biomaterials show inductive properties.'
Relevant doc(s): {'31715818'}


## Query rewrite


In [3]:
from query_rewrite import rewrite_query

rewritten = rewrite_query(query)
print(f"Original:  {query}")
print(f"Rewritten: {rewritten}")


Original:  0-dimensional biomaterials show inductive properties.
Rewritten: Materials with zero dimensions exhibiting inductive behavior.


## Multi-query — expand into several phrasings, fuse the results


In [4]:
from multi_query import generate_queries, multi_query_search

variants = generate_queries(query, n=3)
print("Generated variants:")
for v in variants:
    print(" -", v)

plain_results = {d for d, _ in dense.search(query, top_k=5)}
multi_results = {d for d, _ in multi_query_search(query, dense, n=3, top_k=5)}
print(f"\nPlain Top-5 hit relevant?      {bool(plain_results & relevant)}")
print(f"Multi-query Top-5 hit relevant? {bool(multi_results & relevant)}")


Generated variants:
 - properties of 0D biomaterials
 - Biomaterials with 0 dimensions and inductive properties
 - Inductive behavior of nanostructured biomaterials



Plain Top-5 hit relevant?      False
Multi-query Top-5 hit relevant? False


## HyDE — embed a hypothetical answer instead of the query


In [5]:
from hyde import generate_hypothetical_document, hyde_search

hypothetical = generate_hypothetical_document(query)
print(f"Hypothetical passage:\n{hypothetical}\n")

hyde_results = {d for d, _ in hyde_search(query, dense, top_k=5)}
print(f"HyDE Top-5 hit relevant? {bool(hyde_results & relevant)}")


Hypothetical passage:
Zero-dimensional biomaterials, also known as nanoparticles and nanostructured materials, have been found to exhibit inductive properties due to their unique physical and chemical characteristics. These properties allow them to interact with biological systems at the molecular level, facilitating interactions between biomaterials and cells. Research has shown that zero-dimensional biomaterials can induce specific cellular responses, such as differentiation and proliferation, making them promising candidates for biomedical applications.



HyDE Top-5 hit relevant? True


## Step-back — ask a more general question first


In [6]:
from step_back import step_back_search

result = step_back_search(query, dense, top_k=5)
print(f"Step-back question: {result['step_back_question']}\n")

step_back_docs = {d for d, _ in result['step_back_results']}
original_docs = {d for d, _ in result['original_results']}
print(f"Step-back Top-5 hit relevant?  {bool(step_back_docs & relevant)}")
print(f"Original Top-5 hit relevant?   {bool(original_docs & relevant)}")


Step-back question: What are the fundamental characteristics of materials that enable them to exhibit inductive properties?

Step-back Top-5 hit relevant?  False
Original Top-5 hit relevant?   False


## What I observed

This ran on the exact query that stumped *every* plain retriever in `02_dense_vs_sparse_retrieval.ipynb` — dense and sparse both missed it in their raw Top-5. Here, plain retrieval, multi-query, and step-back **all still missed it**, but **HyDE found it**: writing a plausible answer passage first and embedding *that* landed closer, in embedding space, to the real supporting document than the original short claim ever did.

That's the real lesson, not a hypothetical one: query transformations are a targeted fix for **specific failure modes**, not a blanket upgrade — HyDE specifically helps when a short query and its answer are phrased very differently, which is exactly the gap between a claim and the technical abstract that supports it.

Each transformation also costs an extra LLM call (or several, for multi-query) before retrieval even starts — worth it only when the plain query demonstrably fails.

## Next

[06 — Retrieval Evaluation](./06_retrieval_evaluation.ipynb)
